## Задание 1

Попробуем применить градиентный спуск для решения реальной задачи линейной регрессии. Наша цель будет состоять в том, чтобы построить простую модель линейной регрессии для прогнозирования продаж некоторого продукта  на основе количества денег, потраченных на рекламу (эти данные вы уже знаете).


Все веса изначально возьмите равные 0, скорость обучения задайте как 0.00005, и обучите на 500000 итераций.


Что нужно сделать:

* Задать X и y как в первом ДЗ
* Выразить функцию для вычисления среднеквадратичной ошибки
* Задать функцию градиентного спуска с параметрами:
    * X
    * y
    * par - веса регрессии 
    * alpha - скорость обучения
    * iterations - количество итераций

* Обучить градиентный спуск с параметрами из задания и вывести итоговые коэффициенты и величину ошибки. Сравните коэффициенты с тем, кто получилось в ДЗ1.

In [2]:
import numpy as np

In [3]:
import pandas as pd
df = pd.read_csv('Advertising.csv')

In [4]:
df.head()

,Unnamed: 0,TV,radio,newspaper,sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


In [5]:
df.drop('Unnamed: 0', axis=1, inplace=True)

In [6]:
X_raw = df.drop('sales', axis=1).values
y = df['sales'].values

In [7]:
X_raw[:5]

array([[230.1,  37.8,  69.2],
       [ 44.5,  39.3,  45.1],
       [ 17.2,  45.9,  69.3],
       [151.5,  41.3,  58.5],
       [180.8,  10.8,  58.4]])

In [8]:
y[:5]

array([22.1, 10.4,  9.3, 18.5, 12.9])

**Сначала попробовала без масштабирования.**
(закомментированный код в ячейке ниже)

In [9]:
m = len(y) # Количество наблюдений
ones = np.ones((m, 1))
X = np.hstack((ones, X_raw)) # Добавляем столбец единиц слева

**Итог:**
- Без масштабирования данных обучающая модель выдает ошибку - "RuntimeWarning: invalid value encountered in subtract
  cur_par = cur_par - alpha * gradient" 
- и считает финальную ошибку (MSE) равную nan
- на итерации 997 произошел взрыв градиента.


**С масштабированием:**

In [16]:
# стандартизация данных
mean = np.mean(X_raw, axis=0)
std = np.std(X_raw, axis=0)
X_scaled = (X_raw - mean) / std

# Добавляем столбец единиц (для w0) к уже отмасштабированным данным
m = len(y)
ones = np.ones((m, 1))
X = np.hstack((ones, X_scaled))

In [17]:
X[:5]

array([[ 1.        ,  0.96985227,  0.98152247,  1.77894547],
       [ 1.        , -1.19737623,  1.08280781,  0.66957876],
       [ 1.        , -1.51615499,  1.52846331,  1.78354865],
       [ 1.        ,  0.05204968,  1.21785493,  1.28640506],
       [ 1.        ,  0.3941822 , -0.84161366,  1.28180188]])

In [18]:
# Инициализация весов нулями (w0, w1, w2, w3)
params = np.zeros(X.shape[1])

In [19]:
params

array([0., 0., 0., 0.])

In [20]:
# Параметры обучения
alpha = 5e-5
iterations = 500000

In [21]:
# Функция для вычисления среднеквадратичной ошибки (MSE)
def mse(X, y, par):
    m = len(y)
    pred = X @ par
    errors = pred - y
    mse = (1 / m) * np.sum(errors ** 2)
    return mse

In [22]:
# Функция градиентного спуска
def gradient_descent(X, y, par, alpha, iterations):
    m = len(y)
    cur_par = par.copy()
    # print(y[:5])
    
    for i in range(iterations):
        pred = X.dot(cur_par)
        error = pred - y
        gradient = (2 / m) * X.T.dot(error)
                
        # Обновляем веса
        cur_par = cur_par - alpha * gradient

        # Проверка на NaN
        if np.isnan(cur_par).any():
            print(f"Внимание! На итерации {i} произошел взрыв градиента.")
            break
                
        # Каждые 50 000 итераций выводим прогресс
        if i % 50000 == 0:
            current_mse = mse(X, y, cur_par)
            print(f"Итерация {i}: MSE = {current_mse:.4f}; "
                  f"grad = ({gradient[0]:.3f}, {gradient[1]:.3f}, "
                  f"{gradient[2]:.3f}, {gradient[3]:.3f})"
            )
            # print(cur_par)
            # print(pred[:5])
            
    return cur_par

In [23]:
# Обучение и вывод результатов
final_weights = gradient_descent(X, y, params, alpha, iterations)
final_mse = mse(X, y, final_weights)

print("\nРезультаты:")
print(f"Финальная ошибка (MSE): {final_mse:.4f}")
print(f"Веса (для нормализованных данных): {final_weights}")

Итерация 0: MSE = 223.6715; grad = (-28.045, -8.142, -5.998, -2.376)
Итерация 50000: MSE = 2.7977; grad = (-0.189, -0.050, -0.071, 0.074)
Итерация 100000: MSE = 2.7841; grad = (-0.001, -0.000, -0.003, 0.003)
Итерация 150000: MSE = 2.7841; grad = (-0.000, -0.000, -0.000, 0.000)
Итерация 200000: MSE = 2.7841; grad = (-0.000, -0.000, -0.000, 0.000)
Итерация 250000: MSE = 2.7841; grad = (-0.000, -0.000, -0.000, 0.000)
Итерация 300000: MSE = 2.7841; grad = (-0.000, -0.000, -0.000, 0.000)
Итерация 350000: MSE = 2.7841; grad = (-0.000, -0.000, -0.000, 0.000)
Итерация 400000: MSE = 2.7841; grad = (-0.000, -0.000, -0.000, 0.000)
Итерация 450000: MSE = 2.7841; grad = (-0.000, -0.000, -0.000, 0.000)

Результаты:
Финальная ошибка (MSE): 2.7841
Веса (для нормализованных данных): [14.0225      3.91925365  2.79206274 -0.02253861]


In [32]:
# Возвращаем веса обратно
w_scaled = final_weights[1:]
b_scaled = final_weights[0]

real_weights = w_scaled / std
real_bias = b_scaled - np.sum(real_weights * mean)

final_real_params = np.hstack(([real_bias], real_weights))
final_real_mse = np.sqrt(final_mse)

print("\nРезультаты:")
print(f"Финальная ошибка (MSE) (неквадратичная): {final_real_mse:.4f}")
print(f"Веса (для реальных данных): {final_real_params}")


Результаты:
Финальная ошибка (MSE) (неквадратичная): 1.6686
Веса (для реальных данных): [ 2.93888937e+00  4.57646455e-02  1.88530017e-01 -1.03749304e-03]


**Сравнение полученного результата с результатом из первого ДЗ**

Веса:
- из 1ДЗ - [ 2.93888937e+00,  4.57646455e-02,  1.88530017e-01, -1.03749304e-03]
- из 2ДЗ - [ 2.93888937e+00  4.57646455e-02  1.88530017e-01 -1.03749304e-03]

Финальная ошибка (MSE):  
- из 1ДЗ - 1.6685701407225697 
- из 2ДЗ - 1,668562255356389

**Результаты идентичные!!!**

**Сравнение предсказаний**

In [24]:
final_pred = X @ final_weights

In [25]:
final_pred[:5]

array([20.52397441, 12.33785482, 12.30767078, 17.59782951, 13.18867186])

In [45]:
y[:5]

array([22.1, 10.4,  9.3, 18.5, 12.9])

In [36]:
from sklearn.metrics import r2_score

In [44]:
r2_score(y, final_pred)

0.8972106381789522

**Вывод:**

Прогностическая модель в обоих случаях дала одинаково высокий результат прогнозирования.

из 1ДЗ - Средняя ошибка прогнозирования составляет 1.6685 и 89.7% случаев модель может объяснить.

из 2ДЗ - Средняя ошибка прогнозирования составляет 2.7841 (1,6685**1/2) и 89.7% случаев модель может объяснить.

Первые пять предсказаний: 
- из 1ДЗ - array([20.52397441, 12.33785482, 12.30767078, 17.59782951, 13.18867186])
- из 2ДЗ - array([20.52397441, 12.33785482, 12.30767078, 17.59782951, 13.18867186])

**Предсказания идентичные!!!**

## Задание 2

Мы с вами посмотрели, как можно полностью самостоятельно, “с нуля”, реализовать Байесовский классификатор с довольно высокой точностью. Теперь вам нужно будет воспроизвести то же самое, но на новых данных, которые содержат электронные письма со спамом и без.

Данные для задания: https://drive.google.com/file/d/1SiIngjkkScdn_fFFmzCrh5B_UjphaBhx/view?usp=sharing

In [ ]:
# ваш код

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd

In [3]:
email_data = pd.read_csv('spam_or_not_spam.csv')
email_data.sample(5)

,email,label
1760,use perl daily headline mailer damian conway p...,0
2894,if this promotion has reached you in error an...,1
1885,url URL date not supplied for uk NUMBER NUMBER...,0
1975,url URL date not supplied the controversial cl...,0
215,hello folks i m new to linux so here goes i ve...,0


### Предобработка данных

In [4]:
email_data_clean = email_data.copy()

In [5]:
email_data_clean.groupby('label').count()

,email
label,
0,2500
1,499


In [6]:
email_data_clean['label'] = email_data_clean['label'].replace({0: 'ham', 1: 'spam'})

In [7]:
email_data_clean.groupby('label').count()

,email
label,
ham,2500
spam,499


In [8]:
email_data_clean['label'].value_counts() / email_data_clean.shape[0] * 100

label
ham     83.333333
spam    16.666667
Name: count, dtype: float64

In [9]:
email_data_clean['email'].head()

0     date wed NUMBER aug NUMBER NUMBER NUMBER NUMB...
1    martin a posted tassos papadopoulos the greek ...
2    man threatens explosion in moscow thursday aug...
3    klez the virus that won t die already the most...
4     in adding cream to spaghetti carbonara which ...
Name: email, dtype: object

In [10]:
email_data_clean['email'] = email_data_clean['email'].str.replace('\W+', ' ').str.replace('\s+', ' ').str.replace('_', ' ').str.strip()

In [11]:
email_data_clean['email'] = email_data_clean['email'].str.lower()

In [12]:
email_data_clean['email'] = email_data_clean['email'].str.split()

In [13]:
email_data_clean.dropna(subset=['email'], inplace=True)

In [14]:
email_data_clean['email'].head()

0    [date, wed, number, aug, number, number, numbe...
1    [martin, a, posted, tassos, papadopoulos, the,...
2    [man, threatens, explosion, in, moscow, thursd...
3    [klez, the, virus, that, won, t, die, already,...
4    [in, adding, cream, to, spaghetti, carbonara, ...
Name: email, dtype: object

In [15]:
email_data_clean.shape

(2999, 2)

### Разделение на обучающую и тестовую выборки

In [16]:
train_data = email_data_clean.sample(frac=0.8,random_state=1).reset_index(drop=True)
test_data = email_data_clean.drop(train_data.index).reset_index(drop=True)
train_data = train_data.reset_index(drop=True)

In [17]:
train_data['label'].value_counts() / train_data.shape[0] * 100

label
ham     83.451438
spam    16.548562
Name: count, dtype: float64

In [18]:
train_data['label'].value_counts()

label
ham     2002
spam     397
Name: count, dtype: int64

In [19]:
train_data.shape

(2399, 2)

In [20]:
test_data.shape

(600, 2)

In [21]:
email_data_clean.shape

(2999, 2)

In [22]:
test_data['label'].value_counts() / test_data.shape[0] * 100

label
spam    83.166667
ham     16.833333
Name: count, dtype: float64

In [23]:
test_data['label'].value_counts()

label
spam    499
ham     101
Name: count, dtype: int64

In [24]:
test_data.sample(5)

,email,label
477,"[take, control, of, your, computer, with, this...",spam
95,"[i, had, the, same, problem, when, installing,...",ham
205,"[an, additional, income, stream, from, your, c...",spam
272,"[does, your, computer, need, an, oil, change, ...",spam
92,"[carlos, luna, wrote, hi, all, does, anyone, k...",ham


### Список слов

In [25]:
vocabulary = list(set(train_data['email'].sum()))

In [26]:
len(vocabulary)

27118

In [27]:
vocabulary[11:20]

['acknowledged',
 'surrounded',
 'universally',
 'find',
 'sloth',
 'railroad',
 'cents',
 'arse',
 '另提供網路特訓']

### Частоты слов

In [28]:
vocab_index = {word: i for i, word in enumerate(vocabulary)}

matrix = []

for email_list in train_data['email']:
    row = [0] * len(vocabulary)
    for word in email_list:
        if word in vocab_index:
            idx = vocab_index[word]
            row[idx] += 1
            
    matrix.append(row)

word_counts_per_email = pd.DataFrame(matrix, columns=vocabulary)
train_data_final = pd.concat([train_data.reset_index(drop=True), word_counts_per_email], axis=1)

In [29]:
print("Размер частотной таблицы:", train_data_final.shape)

Размер частотной таблицы: (2399, 27120)


In [30]:
train_data_final.sample(5)

,email,label,disputed,brilliant,uncomitted,finessed,orchard,arbitarily,premise,bacon,...,feelgood,activation,judo,workgroups,reg,theorists,backyard,considerable,vulnerabilities,milfs
491,"[url, url, date, not, supplied, this, time, ar...",ham,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1805,"[i, wonder, if, the, focus, of, spambayes, oug...",ham,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1656,"[once, upon, a, time, angles, wrote, fyi, this...",ham,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
300,"[once, upon, a, time, brian, wrote, i, have, a...",ham,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
545,"[url, url, date, number, number, numbertnumber...",ham,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Переменные для формулы Байеса

In [31]:
alpha = 1

In [32]:
Nvoc = len(train_data_final.columns[2:])

In [33]:
Nvoc

27118

In [34]:
spam_count = train_data_final['label'].iloc[:, 0].value_counts()['spam']

In [35]:
spam_count

np.int64(397)

In [36]:
Pspam = spam_count / train_data_final.shape[0]

In [37]:
Pspam

np.float64(0.16548561900791997)

In [38]:
ham_count = train_data_final['label'].iloc[:, 0].value_counts()['ham']

In [39]:
ham_count

np.int64(2002)

In [40]:
Pham = ham_count / train_data_final.shape[0]

In [41]:
Pham

np.float64(0.8345143809920801)

In [42]:
train_data_final.loc[train_data_final['label'].iloc[:, 0] == 'spam', 'email'].iloc[:, 0]

2       [dear, sir, madam, if, you, are, fed, up, of, ...
12      [hello, jm, spamassassin, taint, orghuman, gro...
13      [html, i, nternette, kendi, ni, zi, daha, i, y...
17      [newsletter, coach, invest, septembre, number,...
24      [here, s, the, hottest, thing, in, dvds, now, ...
                              ...                        
2365    [financial, power, you, can, depend, on, very,...
2367    [take, control, of, your, computer, with, this...
2373    [fortunes, are, literally, being, made, in, th...
2376    [lowest, rates, available, for, term, life, in...
2385    [boost, your, reception, on, any, cell, phone,...
Name: email, Length: 397, dtype: object

In [43]:
Nspam = train_data_final.loc[train_data_final['label'].iloc[:, 0] == 'spam', 'email'].iloc[:, 0].apply(len).sum()

In [44]:
Nspam

np.int64(137855)

In [45]:
train_data_final.loc[train_data_final['label'].iloc[:, 0] == 'ham', 'email'].iloc[:, 0]

0       [url, url, date, number, number, numbertnumber...
1       [followups, to, spambayes, url, please, unless...
3       [url, url, date, number, number, numbertnumber...
4       [use, perl, daily, newsletter, in, this, issue...
5       [date, fri, number, aug, number, number, numbe...
                              ...                        
2394    [with, our, telecoms, partner, bumblebee, don,...
2395    [on, thu, sep, number, number, at, number, num...
2396    [i, wrote, a, script, some, time, ago, to, try...
2397    [url, url, date, number, number, numbertnumber...
2398    [url, url, date, number, number, numbertnumber...
Name: email, Length: 2002, dtype: object

In [46]:
Nham = train_data_final.loc[train_data_final['label'].iloc[:, 0] == 'ham', 'email'].iloc[:, 0].apply(len).sum()

In [47]:
Nham

np.int64(422542)

In [48]:
import numpy as np

In [49]:
def p_w_spam(word):
    if word in train_data_final.columns[2:]:
        subset = train_data_final.loc[train_data_final['label'].iloc[:, 0] == 'spam', word]
        
        # защита от дубликатов колонок
        if isinstance(subset, pd.DataFrame):
            subset = subset.iloc[:, 0]  
            
        # если список - берем первый элемент, иначе оставляем как есть
        def clean_val(x):
            if isinstance(x, list):
                return x[0] if len(x) > 0 else 0
            return x
            
        clean_series = subset.apply(clean_val)
        clean_series = pd.to_numeric(clean_series, errors='coerce').fillna(0)
        
        numeric_sum = clean_series.sum()
        
        return (numeric_sum + alpha) / (Nspam + alpha * Nvoc)
    else:
        return 1

In [50]:
def p_w_ham(word):
    if word in train_data_final.columns[2:]:
        subset = train_data_final.loc[train_data_final['label'].iloc[:, 0] == 'ham', word]
        
        if isinstance(subset, pd.DataFrame):
            subset = subset.iloc[:, 0]
            
        def clean_val(x):
            if isinstance(x, list):
                return x[0] if len(x) > 0 else 0
            return x
            
        clean_series = subset.apply(clean_val)
        clean_series = pd.to_numeric(clean_series, errors='coerce').fillna(0)
        
        numeric_sum = clean_series.sum()
        
        return (numeric_sum + alpha) / (Nham + alpha * Nvoc)
    else:
        return 1

### Готовим алгоритм классификации

In [51]:
import math

In [52]:
def classify(message):
    log_p_spam = math.log(Pspam)
    log_p_ham = math.log(Pham)

    for word in message:
        log_p_spam += math.log(p_w_spam(word))
        log_p_ham += math.log(p_w_ham(word))

    if log_p_ham > log_p_spam:
        return 'ham'
    elif log_p_spam > log_p_ham:
        return 'spam'
    else:
        return 'ham'

### Используем тестовые данные. (Предсказываем)

In [53]:
test_data['predicted'] = test_data['email'].apply(classify)

In [54]:
test_data.sample(7)

,email,label,predicted
226,"[this, messsage, was, sent, using, the, trial,...",spam,spam
488,"[professional, effective, debt, collection, se...",spam,spam
258,"[wind, up, your, mates, today, please, visit, ...",spam,spam
320,"[attn, i, am, bulawa, mulete, jr, the, son, of...",spam,spam
69,"[url, loo, rolls, of, honour, and, the, invisi...",ham,ham
377,"[dear, ilug, url, life, partners, are, life, i...",spam,spam
551,"[fortunes, are, literally, being, made, in, th...",spam,spam


In [55]:
correct = (test_data['predicted'] == test_data['label']).sum() / test_data.shape[0] * 100

In [56]:
test_data.loc[test_data['predicted'] != test_data['label']]

,email,label,predicted
172,"[number, number, number, number, number, numbe...",spam,ham
343,"[x, m, a, h, c, x, u, a, a, x, u, d, i, i, z, ...",spam,ham
344,"[this, url, email, is, sponsored, by, osdn, ti...",spam,ham
359,"[dear, sir, madam, i, got, this, email, id, fr...",spam,ham
363,"[r, v, r, f, i, i, c, u, c, j, c, i, url, i, c...",spam,ham
407,[],spam,ham
417,"[welcome, to, ghetto, america, yes, you, heard...",spam,ham
420,"[r, v, r, f, i, i, c, u, c, j, c, i, url, i, c...",spam,ham
423,"[r, v, r, f, i, i, c, u, c, j, c, i, url, i, c...",spam,ham
424,"[r, v, r, f, i, i, c, u, c, j, c, i, url, i, c...",spam,ham


In [57]:
correct

np.float64(97.83333333333334)